In [1]:
import time
from robot import MotomanRobot
from planning_scene import PlanningScene
import numpy as np
import hppfcl

In [2]:
# test the robot with the scene
# add environment collisions
pcd_total = []
# shelf-bottom
num_points = 1000
position = np.array([0.85, 0, 0.5])
half_size = np.array([0.175, 0.5, 0.5])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-top
num_points = 1000
position = np.array([0.85, 0, 1.42])
half_size = np.array([0.175, 0.5, 0.025])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-left
num_points = 1000
position = np.array([0.85, -0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-right
num_points = 1000
position = np.array([0.85, 0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-back
num_points = 1000
position = np.array([1.0, 0, 1.2])
half_size = np.array([0.025, 0.5, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
pcd_total = np.concatenate(pcd_total, axis=0)


torso_b1 = ["torso_joint_b1"]
left = [
    "arm_left_joint_1_s",
    "arm_left_joint_2_l",
    "arm_left_joint_3_e",
    "arm_left_joint_4_u",
    "arm_left_joint_5_r",
    "arm_left_joint_6_b",
    "arm_left_joint_7_t",
]
right = [
    "arm_right_joint_1_s",
    "arm_right_joint_2_l",
    "arm_right_joint_3_e",
    "arm_right_joint_4_u",
    "arm_right_joint_5_r",
    "arm_right_joint_6_b",
    "arm_right_joint_7_t",
]
robot_joint_names = torso_b1 + left + right
robot = MotomanRobot(selected_joint_names=robot_joint_names)
# scene = PlanningScene(robot, scene_pcd=pcd_total)
scene = PlanningScene(robot, scene_pcd=None)
scene.update_scene_pcd(pcd_total)



In [3]:
start_time = time.time()
robot_joint_angles = np.random.uniform(robot.selected_joint_limits[:,0], robot.selected_joint_limits[:,1])
robot.set_selected_joint_values(robot_joint_angles)
collision_results = scene.compute_collision_total(security_margin=0.1, full=True)


In [4]:

def robot_compute_distance_total(robot, dist_margin: float = 0.001, full: bool = False) -> list:
    """
    compute the distance between all the collision pairs.
    NOTE: this is much slower than the compute_collision_total
    """
    # TODO: add early stopping

    distance_results = []  # store the results of the collision, link1, link2, geom1_i, geom2_i, collision result
    for i in range(len(robot.collision_pairs)):
        link_1, link_2 = robot.collision_pairs[i]
        for obj1_i in range(len(robot.robot_link_name_to_fcl_objs[link_1])):
            for obj2_i in range(len(robot.robot_link_name_to_fcl_objs[link_2])):
                dis_result = hppfcl.DistanceResult()
                distance = hppfcl.distance(robot.robot_link_name_to_fcl_objs[link_1][obj1_i],
                                            robot.robot_link_name_to_fcl_objs[link_2][obj2_i],
                                            robot.collision_pair_to_distance_req[(i, obj1_i, obj2_i)],
                                            dis_result)
                                        #    self.collision_pair_to_distance_res[(i, obj1_i, obj2_i)])
                if distance < dist_margin:
                    distance_result = dis_result
                    # distance_result = self.collision_pair_to_distance_res[(i, obj1_i, obj2_i)]
                    # distance_result = copy.deepcopy(distance_result)
                    distance_results.append((link_1, link_2, obj1_i, obj2_i, distance_result))
                else:
                    if full:
                        distance_result = dis_result
                        # distance_result = self.collision_pair_to_distance_res[(i, obj1_i, obj2_i)]
                        # distance_result = copy.deepcopy(distance_result)
                        distance_results.append((link_1, link_2, obj1_i, obj2_i, distance_result))
    return distance_results

def robot_compute_collision_total(robot, security_margin: float = 0.001, full: bool = False) -> list:
    """
    compute the collision between all the collision pairs.
    """
    # TODO: add early stopping

    collision_results = []  # store the results of the collision, link1, link2, geom1_i, geom2_i, collision result
    for i in range(len(robot.collision_pairs)):
        link_1, link_2 = robot.collision_pairs[i]
        for obj1_i in range(len(robot.robot_link_name_to_fcl_objs[link_1])):
            for obj2_i in range(len(robot.robot_link_name_to_fcl_objs[link_2])):
                # set the safety margin
                robot.collision_pair_to_collision_req[(i, obj1_i, obj2_i)].distance_upper_bound = security_margin
                robot.collision_pair_to_collision_req[(i, obj1_i, obj2_i)].security_margin = 0.001

                col_result = hppfcl.CollisionResult()
                collision = hppfcl.collide(robot.robot_link_name_to_fcl_objs[link_1][obj1_i],
                                            robot.robot_link_name_to_fcl_objs[link_2][obj2_i],
                                            robot.collision_pair_to_collision_req[(i, obj1_i, obj2_i)],
                                            col_result)
                                        #    self.collision_pair_to_collision_res[(i, obj1_i, obj2_i)])
                if collision:
                    collision_result = col_result#self.collision_pair_to_collision_res[(i, obj1_i, obj2_i)]
                    # collision_result = copy.deepcopy(collision_result)
                    collision_results.append((link_1, link_2, obj1_i, obj2_i, collision_result))
                else:
                    if full:
                        collision_result = col_result#self.collision_pair_to_collision_res[(i, obj1_i, obj2_i)]
                        # collision_result = copy.deepcopy(collision_result)
                        collision_results.append((link_1, link_2, obj1_i, obj2_i, collision_result))
    return collision_results

In [5]:

def compute_collision_total(scene, security_margin: float = 0.001, full: bool = False) -> list:
    """
    compute the total collision in the planning scene, including self collision and collision with the scene
    the scene collision is named as "scene"
    :return: the total collision results
    """
    # * self collision
    self_collision_results = robot_compute_collision_total(robot, security_margin, full=full)
    # * scene collision
    collision_results = []  # store the results of the collision, link1, link2, geom1_i, geom2_i, collision result
    for link in scene.robot.robot_link_names:
        for obj_i in range(len(scene.robot.robot_link_name_to_fcl_objs[link])):
            col_result = hppfcl.CollisionResult()
            scene.robot.robot_link_to_collision_req[(link,obj_i)].distance_upper_bound = security_margin
            scene.robot.robot_link_to_collision_req[(link,obj_i)].security_margin = 0.001#security_margin
            collision = hppfcl.collide(scene.robot.robot_link_name_to_fcl_objs[link][obj_i], scene.octree_obj,
                                        scene.robot.robot_link_to_collision_req[(link,obj_i)],
                                        col_result)
                                    #    self.robot.robot_link_to_collision_res[(link,obj_i)])
            # NOTE: collision with the environment (octree/octomap) is using robot.robot_link_to_collision_req
            #       self collision is using robot.collision_pair_to_collision_req
            # TODO: renaming robot.robot_link_to_colllision_res to be more understandable and consistent with self_collision or collision with env

            if collision:
                collision_result = col_result#self.robot.robot_link_to_collision_res[(link,obj_i)]
                collision_results.append((link, "scene", obj_i, 0, collision_result))
            else:
                if full:
                    collision_result = col_result#self.robot.robot_link_to_collision_res[(link,obj_i)]
                    collision_results.append((link, "scene", obj_i, 0, collision_result))
                
    return self_collision_results + collision_results

In [6]:
def compute_distance_total(scene, dist_margin: float = 0.001, full: bool = False) -> list:
    # * self distance
    self_distance_results = scene.robot.compute_distance_total(dist_margin)
    # * scene distance
    distance_results = []
    for link in scene.robot.robot_link_names:
        for obj_i in range(len(scene.robot.robot_link_name_to_fcl_objs[link])):
            dis_result = hppfcl.DistanceResult()
            distance = hppfcl.distance(scene.robot.robot_link_name_to_fcl_objs[link][obj_i], scene.octree_obj,
                                        scene.robot.robot_link_to_distance_req[(link,obj_i)],
                                        dis_result)
                                    #    self.robot.robot_link_to_distance_res[(link,obj_i)])
            if distance < dist_margin:
                distance_result = dis_result
                # distance_result = self.robot.robot_link_to_distance_res[(link,obj_i)]
                distance_results.append((link, "scene", obj_i, 0, distance_result))
            else:
                if full:
                    distance_result = dis_result
                    # distance_result = self.robot.robot_link_to_distance_res[(link,obj_i)]
                    distance_results.append((link, "scene", obj_i, 0, distance_result))
    return self_distance_results + distance_results



In [7]:
scene.robot.robot_link_to_collision_req[('left_pad',0)].security_margin

0.1

In [8]:
start_time = time.time()
while True:
    robot_joint_angles = np.random.uniform(robot.selected_joint_limits[:,0], robot.selected_joint_limits[:,1])
    robot.set_selected_joint_values(robot_joint_angles)
    res = scene.compute_collision_total(dist_upper_bound=0.1, security_margin=0.001, full=False)
    if len(res) > 0:
        break
scene.visualize();


In [9]:
import open3d as o3d
import open3d.visualization as o3d_vis
# from open3d.web_visualizer import draw

In [10]:
# do box-box collision and visualize the point and normal vector to understand
half_size = 0.25
geom1 = hppfcl.Box(half_size*2, half_size*2, half_size*2)
geom2 = hppfcl.Box(half_size*2, half_size*2, half_size*2)
pose1 = np.eye(4)
T = hppfcl.Transform3f(pose1[:3,:3], pose1[:3,3])
obj1 = hppfcl.CollisionObject(geom1, T)
pose2 = np.eye(4)
pose2[:3,3] = np.array([0, 0.7, 0])
T = hppfcl.Transform3f(pose2[:3,:3], pose2[:3,3])
obj2 = hppfcl.CollisionObject(geom2, T)

o3d_box_pose = np.eye(4)
o3d_box_pose[0,3] = o3d_box_pose[0,3] - half_size
o3d_box_pose[1,3] = o3d_box_pose[1,3] - half_size
o3d_box_pose[2,3] = o3d_box_pose[2,3] - half_size

vis_box1 = o3d.geometry.TriangleMesh.create_box(half_size*2, half_size*2, half_size*2).transform(pose1@o3d_box_pose)
vis_box2 = o3d.geometry.TriangleMesh.create_box(half_size*2, half_size*2, half_size*2).transform(pose2@o3d_box_pose)

vis_box1 = o3d.geometry.LineSet.create_from_triangle_mesh(vis_box1).paint_uniform_color([1,0,0])
vis_box2 = o3d.geometry.LineSet.create_from_triangle_mesh(vis_box2).paint_uniform_color([0,0,1])

# vis_box1.material.material_name = 'defaultLit'

# mat1 = o3d_vis.rendering.MaterialRecord()
# # mat1 = o3d_vis.Material()
# mat1.shader = 'defaultLitSSR'
# mat1.base_color = [0.8, 0, 0, 0.1]
# mat2 = o3d_vis.rendering.MaterialRecord()
# mat2.shader = 'defaultLitSSR'
# mat2.base_color = [0, 0, 0.8, 0.1]

# mat_box = o3d.visualization.rendering.MaterialRecord()
# # mat_box.shader = 'defaultLitTransparency'
# mat_box.shader = 'defaultLitSSR'
# mat_box.base_color = [0.467, 0.467, 0.467, 0.2]
# mat_box.base_roughness = 0.0
# mat_box.base_reflectance = 0.0
# mat_box.base_clearcoat = 1.0
# mat_box.thickness = 1.0
# mat_box.transmission = 1.0
# mat_box.absorption_distance = 10
# mat_box.absorption_color = [0.5, 0.5, 0.5]

# geoms = [{'name': 'box1', 'geometry': vis_box1, 'material': mat1},
#          {'name': 'box2', 'geometry': vis_box2, 'material': mat2}]
# o3d.visualization.draw(geoms)
# draw(geoms)
o3d.visualization.draw_geometries([vis_box1, vis_box2])


In [11]:
col_request = hppfcl.CollisionRequest()
col_request.security_margin = 0.001
col_result = hppfcl.CollisionResult()
hppfcl.collide(obj1, obj2, col_request, col_result)




0

In [12]:
arrow = o3d.geometry.TriangleMesh.create_arrow(cylinder_radius=0.01, cone_radius=0.02, cylinder_height=0.1, cone_height=0.1).rotate(R, center=[0,0,0])
frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1, origin=[0,0,0])
o3d.visualization.draw_geometries([arrow, frame])

NameError: name 'R' is not defined

In [ ]:
print('number of contacts: ', col_result.numContacts())
contact = col_result.getContact(0)
normal = np.array(contact.normal)  # according to the doc, this points from obj1 to obj2
distance = -contact.penetration_depth
print('distance: ', distance)
print('normal: ', normal)
pt = np.array(contact.pos)  # visualize this
print('pos: ', pt)
pt1 = pt - normal * 0.5 * distance
pt2 = pt + normal * 0.5 * distance

pt_o3d = o3d.geometry.TriangleMesh.create_sphere(radius=0.02).translate(pt).paint_uniform_color([0,1,0])
pt1_o3d = o3d.geometry.TriangleMesh.create_sphere(radius=0.02).translate(pt1).paint_uniform_color([1,0,0])
pt2_o3d = o3d.geometry.TriangleMesh.create_sphere(radius=0.02).translate(pt2).paint_uniform_color([0,0,1])


R = np.eye(3)
R[:, 2] = contact.normal
R[:, 0] = np.cross(contact.normal, [0, 0, 1])
R[:, 0] /= np.linalg.norm(R[:, 0])
R[:, 1] = np.cross(R[:, 2], R[:, 0])
arrow = o3d.geometry.TriangleMesh.create_arrow(cylinder_radius=0.01, cone_radius=0.02, cylinder_height=0.1, cone_height=0.1).rotate(R, center=[0,0,0]).translate(contact.pos).paint_uniform_color([1, 0, 0])

o3d.visualization.draw_geometries([vis_box1, vis_box2, pt_o3d, pt1_o3d, pt2_o3d, arrow])



number of contacts:  0


ValueError: The number of contacts is zero. No Contact can be returned.

In [17]:
dist_request = hppfcl.DistanceRequest()
print(dist_request.enable_nearest_points)
col_request = hppfcl.CollisionRequest()
print(col_request.enable_distance_lower_bound)

False
False


In [ ]:
# check for distance
dist_request = hppfcl.DistanceRequest()
dist_request.enable_nearest_points = True
dist_result = hppfcl.DistanceResult()
distance = hppfcl.distance(obj1, obj2, dist_request, dist_result)
min_distance = dist_result.min_distance
nearest_pt1 = np.array(dist_result.getNearestPoint1())
nearest_pt2 = np.array(dist_result.getNearestPoint2())
normal = np.array(dist_result.normal)
print('min_distance: ', min_distance)
print('normal: ', normal)
print('nearest_pt1: ', nearest_pt1)
print('nearest_pt2: ', nearest_pt2)


min_distance:  0.19999999499999999
normal:  [ 0. -1.  0.]
nearest_pt1:  [ 0.    0.25 -0.25]
nearest_pt2:  [ 0.    0.45 -0.25]


In [ ]:
collision_results = scene.compute_collision_total(dist_upper_bound=0.1, security_margin=0.001, full=True)

In [ ]:
import open3d as o3d

In [ ]:
# check results

o3d_vis = scene.visualize(False)
for link_1, link_2, obj1_i, obj2_i, collision_result in collision_results:
    if not collision_result.isCollision():
        continue
    print('number of contacts: ', collision_result.numContacts())
    print("nearest point: ")
    # check the contact
    for i in range(collision_result.numContacts()):
        contact = collision_result.getContact(i)
        print('distance: ', -contact.penetration_depth)  # the negative is the distance
        print('link1: ', link_1)
        print('link2: ', link_2)
        print('normal: ', contact.normal)
        print('position: ', contact.pos)
        print('=====================')
        # visualize the contact direction
        R = np.eye(3)
        R[:, 2] = contact.normal
        R[:, 0] = np.cross(contact.normal, [0, 0, 1])
        R[:, 0] /= np.linalg.norm(R[:, 0])
        R[:, 1] = np.cross(R[:, 2], R[:, 0])
        arrow = o3d.geometry.TriangleMesh.create_arrow(cylinder_radius=0.01, cone_radius=0.02, cylinder_height=0.1, cone_height=0.1).rotate(R).translate(contact.pos).paint_uniform_color([1, 0, 0]).compute_vertex_normals()
        o3d_vis.append(arrow)
o3d.visualization.draw_geometries(o3d_vis)

number of contacts:  1
nearest point: 
distance:  0.0005622419021703926
link1:  base
link2:  arm_left_link_5_r
normal:  [ 0.8272702  -0.54845878  0.12172502]
position:  [ 0.07180428 -0.09807166  0.87227772]
number of contacts:  1
nearest point: 
distance:  0.0004892983191389233
link1:  torso_link_b1
link2:  arm_left_link_4_u
normal:  [ 0.56893734 -0.8222412  -0.01515619]
position:  [ 0.06603977 -0.09787691  0.90088159]
number of contacts:  1
nearest point: 
distance:  -0.024109161476072994
link1:  torso_link_b1
link2:  arm_left_link_5_r
normal:  [ 0.98346176 -0.1808058  -0.01059382]
position:  [ 0.11480063 -0.03217581  0.91797359]


In [ ]:
arrow = o3d.geometry.TriangleMesh.create_arrow(cylinder_radius=0.01, cone_radius=0.02, cylinder_height=0.1, cone_height=0.1)
frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1, origin=[0, 0, 0])
o3d.visualization.draw_geometries([arrow, frame])
# arrow direction: z axis

In [ ]:
# check results
for link_1, link_2, obj1_i, obj2_i, collision_result in collision_results:
    if not collision_result.isCollision():
        continue
    print('number of contacts: ', collision_result.numContacts())
    print("nearest point: ")
    # check the contact
    for i in range(collision_result.numContacts()):
        contact = collision_result.getContact(i)
        print('distance: ', -contact.penetration_depth)  # the negative is the distance
        print('link1: ', link_1)
        print('link2: ', link_2)
        print('normal: ', contact.normal)
        print('position: ', contact.pos)
        print('=====================')


number of contacts:  1
nearest point: 
distance:  0.0005622419021703926
link1:  base
link2:  arm_left_link_5_r
normal:  [ 0.8272702  -0.54845878  0.12172502]
position:  [ 0.07180428 -0.09807166  0.87227772]
number of contacts:  1
nearest point: 
distance:  0.0004892983191389233
link1:  torso_link_b1
link2:  arm_left_link_4_u
normal:  [ 0.56893734 -0.8222412  -0.01515619]
position:  [ 0.06603977 -0.09787691  0.90088159]
number of contacts:  1
nearest point: 
distance:  -0.024109161476072994
link1:  torso_link_b1
link2:  arm_left_link_5_r
normal:  [ 0.98346176 -0.1808058  -0.01059382]
position:  [ 0.11480063 -0.03217581  0.91797359]


In [13]:
# distance_results = compute_distance_total(scene, dist_margin=0.1, full=True)
distance_results = scene.compute_distance_total(dist_margin=0.1, full=True)

In [15]:
# check results
for link_1, link_2, obj1_i, obj2_i, distance_result in distance_results:
    min_distance = distance_result.min_distance
    if min_distance >= 0.1:
        continue
    print('link1: ', link_1)
    print('link2: ', link_2)
    print('obj1_i: ', obj1_i)
    print('obj2_i: ', obj2_i)
    print('distance: ', min_distance)
    print("nearest point1: ")
    point1 = distance_result.getNearestPoint1()
    print(point1)
    point2 = distance_result.getNearestPoint2()
    print("nearest point2: ")
    print(point2)
    print('normal: ', distance_result.normal)
    print('=====================')


link1:  torso_link_b1
link2:  arm_left_link_2_l
obj1_i:  0
obj2_i:  0
distance:  0.044625948811310365
nearest point1: 
[ 0.11368411 -0.15360315  1.19668905]
nearest point2: 
[ 0.15806951 -0.15823038  1.19669656]
normal:  [1.0173e-320 1.4348e-320 0.0000e+000]
link1:  torso_link_b1
link2:  arm_right_link_2_l
obj1_i:  0
obj2_i:  0
distance:  0.04419537908194297
nearest point1: 
[-0.13661305 -0.0667704   1.22063602]
nearest point2: 
[-0.18057172 -0.06220238  1.22064644]
normal:  [5.044e-321 5.543e-321 0.000e+000]
link1:  torso_link_b1
link2:  arm_right_link_3_e
obj1_i:  0
obj2_i:  0
distance:  0.0789515660909963
nearest point1: 
[-0.12665725  0.01706847  1.05471735]
nearest point2: 
[-0.20393584  0.03318052  1.05338617]
normal:  [2.678e-321 7.727e-321 0.000e+000]
link1:  arm_left_link_1_s
link2:  arm_left_link_3_e
obj1_i:  0
obj2_i:  0
distance:  0.09352063670153035
nearest point1: 
[ 0.34206534 -0.06912943  1.22154539]
nearest point2: 
[ 0.43117437 -0.07785807  1.24855465]
normal:  [8.745

In [ ]:
dist_upper_bound = 0.1
distance_results = scene.compute_collision_min_dist_total(dist_upper_bound=0.1, full=True)

In [19]:
# check results
for i in range(len(distance_results)):
    link_1, link_2, obj1_i, obj2_i, distance_result = distance_results[i]
    _, _, _, _, collision_result = collision_results[i]
    min_distance = distance_result.min_distance
    if min_distance >= dist_upper_bound:
        continue
    print('link1: ', link_1)
    print('link2: ', link_2)
    print('obj1_i: ', obj1_i)
    print('obj2_i: ', obj2_i)
    print('distance: ', min_distance)
    print("nearest point1: ")
    point1 = distance_result.getNearestPoint1()
    print(point1)
    point2 = distance_result.getNearestPoint2()
    print("nearest point2: ")
    print(point2)
    # check norm and point
    normal = distance_result.normal
    print('normal: ', normal)
    normal_point = point2 - point1
    print('normal - normal_point: ', np.linalg.norm(normal - normal_point/np.linalg.norm(normal_point)))
    print('=====================')

link1:  base
link2:  arm_left_link_3_e
obj1_i:  0
obj2_i:  0
distance:  0.0801126274529832
nearest point1: 
[-0.03699313 -0.12184692  0.52606036]
nearest point2: 
[-0.0474213  -0.20027431  0.53864732]
normal:  [1.087e-321 1.996e-321 0.000e+000]
normal - normal_point:  1.0
link1:  base
link2:  arm_left_link_4_u
obj1_i:  0
obj2_i:  0
distance:  0.001541237078735188
nearest point1: 
[ 0.05023631 -0.09762857  0.5699971 ]
nearest point2: 
[ 0.05101764 -0.09890463  0.5703667 ]
normal:  [1.655e-321 1.324e-321 0.000e+000]
normal - normal_point:  0.9999999999999999
link1:  base
link2:  arm_left_link_5_r
obj1_i:  0
obj2_i:  0
distance:  0.0
nearest point1: 
[ 0.05524015 -0.09691395  0.56091878]
nearest point2: 
[ 0.05260182 -0.0947825   0.56014684]
normal:  [1.136e-321 7.796e-321 0.000e+000]
normal - normal_point:  1.0
link1:  base
link2:  arm_left_link_6_b
obj1_i:  0
obj2_i:  0
distance:  0.05883043896548708
nearest point1: 
[ 0.09838426 -0.02717487  0.57646465]
nearest point2: 
[ 0.13412695 -0

In [20]:
point1 = np.array([-0.37460699,  0.42765569,  1.31923366])
point2 = np.array([-0.34604712,  0.41734595,  1.30683017])
normal = np.array([-0.6435354,  -0.76442385, -0.18950094])
normal_point = point2 - point1
print('distance: ', np.linalg.norm(normal_point))
normal_point = normal_point / np.linalg.norm(normal_point)
print('normal.norm: ', np.linalg.norm(normal))
print('normal_point: ', normal_point)

distance:  0.03279944324930836
normal.norm:  1.0170507557456836
normal_point:  [ 0.87074252 -0.31432668 -0.3781616 ]


In [21]:
scene.visualize()

[TriangleMesh with 194 points and 384 triangles.,
 TriangleMesh with 579 points and 1154 triangles.,
 TriangleMesh with 732 points and 1460 triangles.,
 TriangleMesh with 803 points and 1590 triangles.,
 TriangleMesh with 451 points and 898 triangles.,
 TriangleMesh with 694 points and 1384 triangles.,
 TriangleMesh with 527 points and 1050 triangles.,
 TriangleMesh with 464 points and 924 triangles.,
 TriangleMesh with 225 points and 446 triangles.,
 TriangleMesh with 732 points and 1460 triangles.,
 TriangleMesh with 803 points and 1590 triangles.,
 TriangleMesh with 451 points and 898 triangles.,
 TriangleMesh with 694 points and 1384 triangles.,
 TriangleMesh with 527 points and 1050 triangles.,
 TriangleMesh with 464 points and 924 triangles.,
 TriangleMesh with 225 points and 446 triangles.,
 TriangleMesh with 10905 points and 21834 triangles.,
 TriangleMesh with 17156 points and 34248 triangles.,
 TriangleMesh with 672 points and 1340 triangles.,
 TriangleMesh with 888 points an